# Week 2 — 多變數微積分與最小變異投資組合

> 本 notebook 屬於「量化數學路線圖」（quant-math-roadmap）開源教學專案。
> 僅供**教育與研究方法論**用途，**不構成投資建議**，任何結果都不代表實際可獲利或可投資的策略。

## 學習目標

- 計算 quadratic 目標函數的梯度與 Hessian。
- 用 Lagrange multiplier 推導等式約束最佳化。
- 實作並解讀最小變異投資組合。
- 觀察噪音共變異數估計如何造成權重不穩定。

## 預估學習時間

約 9–11 小時。

## 先備概念

- 偏微分與梯度
- Week 1 的共變異數與 quadratic form

## 外部學習資源

- [MIT OpenCourseWare 18.02SC Multivariable Calculus](https://ocw.mit.edu/courses/18-02sc-multivariable-calculus-fall-2010/)
- [NTU OpenCourseWare 基礎財金素養](https://ocw.aca.ntu.edu.tw/courses/110S204)

> 外部資源僅供參考連結；本專案不重製任何受版權保護的課程材料。

## 概念說明

### 梯度與 Hessian

對 $f(w) = w^\top\Sigma w$（$\Sigma$ 對稱），有

$$ \nabla f(w) = 2\Sigma w, \qquad \nabla^2 f(w) = 2\Sigma. $$

若 $\Sigma$ 是 PSD，則 Hessian 是 PSD，$f$ 為**凸函數**——這保證最小化問題有良好、唯一的解。

### 最小變異投資組合

問題為：

$$ \min_w\ w^\top\Sigma w \quad \text{s.t.}\quad \mathbf{1}^\top w = 1. $$

用 Lagrangian $L(w,\lambda) = w^\top\Sigma w - \lambda(\mathbf{1}^\top w - 1)$，對 $w$ 求導並令其為零，可得封閉解

$$ w^\* = \frac{\Sigma^{-1}\mathbf{1}}{\mathbf{1}^\top\Sigma^{-1}\mathbf{1}}. $$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from quant_math_roadmap.data import SyntheticConfig, generate_correlated_returns
from quant_math_roadmap.finance.metrics import covariance_matrix
from quant_math_roadmap.finance.portfolio import (
    equal_weights, minimum_variance_portfolio,
    portfolio_variance, shrinkage_covariance,
)
from quant_math_roadmap.math.optimization import (
    quadratic_gradient, quadratic_hessian,
)

config = SyntheticConfig(n_assets=6, n_periods=504, seed=7,
                         average_correlation=0.45)
returns = generate_correlated_returns(config)
cov = covariance_matrix(returns).to_numpy()
cov.shape

### 數值梯度 vs 解析梯度

In [ ]:
w0 = equal_weights(6)
analytic = quadratic_gradient(cov, w0)

# 用有限差分驗證解析梯度
eps = 1e-6
numeric = np.zeros_like(w0)
for i in range(len(w0)):
    step = np.zeros_like(w0)
    step[i] = eps
    f_plus = portfolio_variance(w0 + step, cov)
    f_minus = portfolio_variance(w0 - step, cov)
    numeric[i] = (f_plus - f_minus) / (2 * eps)

print('解析梯度:', np.round(analytic, 6))
print('數值梯度:', np.round(numeric, 6))
print('最大誤差:', np.max(np.abs(analytic - numeric)))

解析梯度 $2\Sigma w$ 與有限差分高度吻合。Hessian 是常數矩陣 $2\Sigma$：

In [ ]:
hessian = quadratic_hessian(cov)
print('Hessian 是否對稱:', np.allclose(hessian, hessian.T))
print('Hessian 最小特徵值:', np.linalg.eigvalsh(hessian).min())
print('-> 非負特徵值代表目標函數為凸函數。')

### 最小變異 vs equal-weight

In [ ]:
mvp = minimum_variance_portfolio(cov)
eq = equal_weights(6)
print('最小變異權重:', np.round(mvp, 4), ' 總和=', round(mvp.sum(), 6))
print('equal-weight  :', np.round(eq, 4))
print()
print(f'最小變異投組變異數 = {portfolio_variance(mvp, cov):.8f}')
print(f'equal-weight 變異數 = {portfolio_variance(eq, cov):.8f}')

在**樣本內**（in-sample），最小變異投組的變異數依定義必然 $\le$ equal-weight。但這不保證**樣本外**也較好——下一段就來檢驗。

### In-sample vs out-of-sample：噪音造成的不穩定

In [ ]:
# 用前半段估計權重，後半段檢驗
half = len(returns) // 2
train, test = returns.iloc[:half], returns.iloc[half:]
cov_train = covariance_matrix(train).to_numpy()
cov_test = covariance_matrix(test).to_numpy()

mvp_train = minimum_variance_portfolio(cov_train)
in_sample = portfolio_variance(mvp_train, cov_train)
out_sample = portfolio_variance(mvp_train, cov_test)
eq_out = portfolio_variance(eq, cov_test)
print(f'最小變異  in-sample 變異數 = {in_sample:.8f}')
print(f'最小變異 out-of-sample 變異數 = {out_sample:.8f}')
print(f'equal-weight out-of-sample 變異數 = {eq_out:.8f}')
print('觀察：out-of-sample 通常比 in-sample 差，有時甚至輸給 equal-weight。')

### Shrinkage：對抗噪音的共變異數估計

In [ ]:
weights_by_shrinkage = {}
for delta in [0.0, 0.2, 0.5, 0.8]:
    cov_shrunk = shrinkage_covariance(train, shrinkage=delta).to_numpy()
    w = minimum_variance_portfolio(cov_shrunk)
    weights_by_shrinkage[delta] = w

fig, ax = plt.subplots(figsize=(8, 4.5))
for delta, w in weights_by_shrinkage.items():
    ax.plot(range(1, 7), w, marker='o', label=f'shrinkage={delta}')
ax.axhline(1 / 6, linestyle='--', label='equal weight')
ax.set_title('shrinkage 強度對最小變異權重的影響')
ax.set_xlabel('資產索引')
ax.set_ylabel('權重')
ax.legend()
plt.show()

shrinkage 越強，權重越被拉向 equal-weight、越不極端。這以一點偏誤換取大幅的穩定度，往往改善樣本外表現。

## 練習

請依序完成以下練習。**基礎練習**鞏固定義，**應用練習**動手寫程式，**反思問題**把數學連結到回測與研究方法論。

> 主 notebook 的程式練習提供可執行的起始碼（starter）。完整參考解答請見 `notebooks/solutions/` 對應的 `_solution` notebook。

### 基礎練習

1. 寫出最小變異問題的 Lagrangian，並說明每一項的意義。
2. 為什麼「Hessian 為 PSD」能保證問題有良好解？
3. 用一句話解釋 shrinkage 是在做什麼取捨。

### 應用練習

In [ ]:
# 應用練習 1：用封閉解公式 w* = (Σ^-1 1)/(1^T Σ^-1 1) 自行計算最小變異權重，
# 並與 minimum_variance_portfolio() 比對。
ones = np.ones(6)
my_mvp = None  # TODO: inv = np.linalg.solve(cov, ones); my_mvp = inv / (ones @ inv)
if my_mvp is not None:
    print('最大誤差:', np.max(np.abs(my_mvp - minimum_variance_portfolio(cov))))

In [ ]:
# 應用練習 2：計算 long-only（不可放空）的最小變異投組，
# 並確認沒有負權重。
long_only = None  # TODO: minimum_variance_portfolio(cov, long_only=True)
if long_only is not None:
    print('最小權重:', long_only.min(), '| 總和:', long_only.sum())

### 反思問題

1. 最小變異投組在樣本內一定贏 equal-weight，樣本外卻不一定。這個現象和 Week 8 的「過度配適 in-sample 期間」有什麼關聯？

## 常見錯誤

- **對噪音很大的共變異數矩陣求逆，得到極端且不穩定的權重。**
- **把 in-sample 的低變異數誤認為樣本外保證。**
- **忘記約束 $\mathbf{1}^\top w = 1$，得到沒有意義的權重。**
- **忽略最佳化結果對共變異數估計誤差非常敏感。**

## 完成本週後，你應該能做到什麼

- [ ] 能寫出並解釋最小變異問題的 Lagrangian。
- [ ] 能用封閉解計算最小變異權重。
- [ ] 能比較 in-sample 與 out-of-sample 變異數。
- [ ] 能說明 shrinkage 如何穩定權重。

## 參考與致謝

- 本 notebook 的所有解說、範例與習題皆為本專案**原創**撰寫。
- 推薦的外部學習資源請見 [`docs/resources.md`](../docs/resources.md)。
- 數學與財務概念筆記請見 [`docs/math/`](../docs/math/) 與 [`docs/finance/`](../docs/finance/)。

### 隱私與免責聲明

- 本 notebook 不含任何真實個人資訊。
- 本 notebook 僅使用可重現的合成資料，不需要網路連線。
- 本 notebook 不對任何策略做出實際投資獲利的宣稱。